In [1]:
# ========================= YOLO11L TRAINING =========================
# Для RTX 4070 Laptop 8GB

# Если ultralytics ещё не установлен:
# pip install -U ultralytics pycocotools pyyaml pandas

import os, shutil, yaml, pandas as pd, torch
from pycocotools.coco import COCO
from ultralytics import YOLO

# ---------- PATHS ----------
BASE_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks"
IMG_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\train_images"
JSON_PATH = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train_fused_deduplicated.json"
FOLDS_CSV = r"X:\Desktop\kaggle\solar_project_cv\notebooks\clean_fused_folds.csv"
YOLO_DATA_DIR = os.path.join(BASE_DIR, "yolo_dataset_l")
MODEL_DIR = r"X:\Desktop\kaggle\solar_project_cv\models"

# ---------- GPU ----------
assert torch.cuda.is_available(), "CUDA GPU not found"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch: 2.5.1+cu118
CUDA: 11.8


In [2]:
# # ---------- CREATE DATASET ----------
# for split in ["train", "val"]:
#     os.makedirs(os.path.join(YOLO_DATA_DIR, "images", split), exist_ok=True)
#     os.makedirs(os.path.join(YOLO_DATA_DIR, "labels", split), exist_ok=True)

# coco = COCO(JSON_PATH)
# df = pd.read_csv(FOLDS_CSV)

# name_to_coco_id = {img["file_name"]: img["id"] for img in coco.dataset["images"]}

# for _, row in df.iterrows():
#     fname = row["file_name"]
#     img_id = name_to_coco_id[fname]
#     split = "val" if row["fold"] == 0 else "train"

#     src = os.path.join(IMG_DIR, fname)
#     dst = os.path.join(YOLO_DATA_DIR, f"images/{split}", fname)
#     if not os.path.exists(dst):
#         shutil.copy2(src, dst)

#     info = coco.loadImgs([img_id])[0]
#     W, H = info["width"], info["height"]
#     anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))

#     label_path = os.path.join(YOLO_DATA_DIR, f"labels/{split}", os.path.splitext(fname)[0] + ".txt")
#     with open(label_path, "w") as f:
#         for ann in anns:
#             x, y, bw, bh = ann["bbox"]
#             xc, yc = (x + bw/2)/W, (y + bh/2)/H
#             f.write(f"0 {xc:.6f} {yc:.6f} {bw/W:.6f} {bh/H:.6f}\n")

In [8]:
yaml_path = os.path.join(YOLO_DATA_DIR, "data.yaml")

os.makedirs(YOLO_DATA_DIR, exist_ok=True)

with open(yaml_path, "w") as f:
    yaml.dump({
        "path": YOLO_DATA_DIR,
        "train": "images/train",
        "val": "images/val",
        "nc": 1,
        "names": ["filament"]
    }, f, sort_keys=False)

print(yaml_path)

X:\Desktop\kaggle\solar_project_cv\notebooks\yolo_dataset_l\data.yaml


In [9]:
# ---------- TRAIN YOLO11L ----------
model = YOLO("yolo11l.pt")

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=1280,
    batch=2,
    device=0,
    workers=4,
    pretrained=True,
    patience=8,
    project=MODEL_DIR,
    name="yolo11l_detector",
    exist_ok=True,
    amp=True,
    plots=True,
    save=True
)

New https://pypi.org/project/ultralytics/8.4.172 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=X:\Desktop\kaggle\solar_project_cv\notebooks\yolo_dataset_l\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, 

In [10]:
import os, cv2, torch, numpy as np, pandas as pd
import segmentation_models_pytorch as smp
from ultralytics import YOLO
from pycocotools import mask as maskUtils
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_SIZE, PAD_RATIO, MASK_THR = 1280, 0.20, 0.50
OUT_DIR = "conf_test_submissions"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_DIR = r"X:\Desktop\kaggle\solar_project_cv\models"
YOLO_PATH = r"X:\Desktop\kaggle\solar_project_cv\models\yolo11l_detector\weights\best.pt"
TEST_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\test\test_images"
CONF_TESTS = [0.30]

yolo = YOLO(YOLO_PATH)
models = []
for fold in range(5):
    m = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"best_seg_resnet34_fold_{fold}.pth"), map_location=DEVICE))
    m.eval()
    models.append(m)

subs = {c: [] for c in CONF_TESTS}
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise ValueError(path)
    if img.ndim == 3: img = img[:, :, 0] if img.shape[2] == 1 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

for fname in tqdm(sorted(os.listdir(TEST_DIR)), desc="Inference"):
    raw = load_gray(os.path.join(TEST_DIR, fname)); H, W = raw.shape
    disk = raw > 15
    clahe = cv2.createCLAHE(2.0, (8, 8))
    enhanced = clahe.apply(raw.astype(np.uint8)); enhanced[~disk] = 0
    yolo_img = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    r = yolo.predict(yolo_img, imgsz=YOLO_SIZE, conf=min(CONF_TESTS), verbose=False)[0]
    if len(r.boxes) == 0: continue
    boxes = r.boxes.xyxy.cpu().numpy(); confs = r.boxes.conf.cpu().numpy()
    order = np.argsort(-confs); image_id = os.path.splitext(fname)[0]
    occupied = {c: np.zeros((H,W), np.uint8) for c in CONF_TESTS}
    counters = {c: 0 for c in CONF_TESTS}

    for bi in order:
        det_conf = confs[bi]
        x1,y1,x2,y2 = boxes[bi].astype(int)
        bw,bh = x2-x1,y2-y1; px,py = int(bw*PAD_RATIO),int(bh*PAD_RATIO)
        x1,y1,x2,y2 = max(0,x1-px),max(0,y1-py),min(W,x2+px),min(H,y2+py)
        crop = enhanced[y1:y2,x1:x2]
        if crop.size == 0: continue

        crop512 = cv2.resize(crop,(512,512),interpolation=cv2.INTER_AREA)
        crop512 = clahe.apply(crop512).astype(np.float32)/255.0
        x = torch.from_numpy(np.stack([crop512,crop512,crop512])).unsqueeze(0).float().to(DEVICE)
        with torch.inference_mode():
            probs = np.stack([torch.sigmoid(m(x))[0,0].cpu().numpy() for m in models])
        mean_prob = probs.mean(axis=0)

        for conf_thr in CONF_TESTS:
            if det_conf < conf_thr: continue
            mask = (mean_prob > MASK_THR).astype(np.uint8)
            mask = cv2.resize(mask,(x2-x1,y2-y1),interpolation=cv2.INTER_NEAREST)
            mask[~disk[y1:y2,x1:x2]] = 0
            full = np.zeros((H,W),np.uint8); full[y1:y2,x1:x2] = mask
            full[occupied[conf_thr] > 0] = 0
            if full.sum() == 0: continue
            occupied[conf_thr][full > 0] = 1
            counters[conf_thr] += 1
            rle = maskUtils.encode(np.asfortranarray(full)); rle["counts"] = rle["counts"].decode("utf-8")
            subs[conf_thr].append({"filament_id":f"{image_id}_{counters[conf_thr]}","segmentation_rle":rle["counts"]})

for conf_thr,data in subs.items():
    df = pd.DataFrame(data, columns=["filament_id","segmentation_rle"])
    assert df["filament_id"].is_unique
    path = os.path.join(OUT_DIR,f"submission_conflarge_{str(conf_thr).replace('.','')}.csv")
    df.to_csv(path,index=False)
    print(f"conf={conf_thr}: {len(df)} instances -> {path}")

Inference: 100%|██████████| 180/180 [01:56<00:00,  1.55it/s]

conf=0.3: 1310 instances -> conf_test_submissions\submission_conflarge_03.csv


In [11]:
results

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002B80BE5ADA0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [13]:
model = YOLO("yolo11l.pt")

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=1536,
    batch=1,
    device=0,
    patience=8,
    project=os.path.join(BASE_DIR, "models"),
    name="yolo11l_1536",
    pretrained=True,
    amp=True,
    workers=4,
    exist_ok=True,
    save=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.172 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=X:\Desktop\kaggle\solar_project_cv\notebooks\yolo_dataset_l\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1536, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, 

In [14]:
del model

In [3]:
yaml_path = os.path.join(YOLO_DATA_DIR, "data.yaml")

model = YOLO("yolo11l.pt")

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=2048,
    batch=1,
    device=0,
    patience=8,
    project=os.path.join(BASE_DIR, "models"),
    name="yolo11l_2048",
    pretrained=True,
    amp=True,
    workers=4,
    exist_ok=True,
    save=True,
    plots=True
)

Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=X:\Desktop\kaggle\solar_project_cv\notebooks\yolo_dataset_l\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=2048, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11l_2048, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

In [4]:
import os, cv2, torch, numpy as np, pandas as pd
import segmentation_models_pytorch as smp
from ultralytics import YOLO
from pycocotools import mask as maskUtils
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_SIZE, PAD_RATIO, MASK_THR = 2048, 0.20, 0.50
OUT_DIR = "conf_test_submissions"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_DIR = r"X:\Desktop\kaggle\solar_project_cv\models"
YOLO_PATH = r"X:\Desktop\kaggle\solar_project_cv\notebooks\models\yolo11l_2048\weights\best.pt"
TEST_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\test\test_images"
CONF_TESTS = [0.30]

yolo = YOLO(YOLO_PATH)
models = []
for fold in range(5):
    m = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"best_seg_resnet34_fold_{fold}.pth"), map_location=DEVICE))
    m.eval()
    models.append(m)

subs = {c: [] for c in CONF_TESTS}
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise ValueError(path)
    if img.ndim == 3: img = img[:, :, 0] if img.shape[2] == 1 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

for fname in tqdm(sorted(os.listdir(TEST_DIR)), desc="Inference"):
    raw = load_gray(os.path.join(TEST_DIR, fname)); H, W = raw.shape
    disk = raw > 15
    clahe = cv2.createCLAHE(2.0, (8, 8))
    enhanced = clahe.apply(raw.astype(np.uint8)); enhanced[~disk] = 0
    yolo_img = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    r = yolo.predict(yolo_img, imgsz=YOLO_SIZE, conf=min(CONF_TESTS), verbose=False)[0]
    if len(r.boxes) == 0: continue
    boxes = r.boxes.xyxy.cpu().numpy(); confs = r.boxes.conf.cpu().numpy()
    order = np.argsort(-confs); image_id = os.path.splitext(fname)[0]
    occupied = {c: np.zeros((H,W), np.uint8) for c in CONF_TESTS}
    counters = {c: 0 for c in CONF_TESTS}

    for bi in order:
        det_conf = confs[bi]
        x1,y1,x2,y2 = boxes[bi].astype(int)
        bw,bh = x2-x1,y2-y1; px,py = int(bw*PAD_RATIO),int(bh*PAD_RATIO)
        x1,y1,x2,y2 = max(0,x1-px),max(0,y1-py),min(W,x2+px),min(H,y2+py)
        crop = enhanced[y1:y2,x1:x2]
        if crop.size == 0: continue

        crop512 = cv2.resize(crop,(512,512),interpolation=cv2.INTER_AREA)
        crop512 = clahe.apply(crop512).astype(np.float32)/255.0
        x = torch.from_numpy(np.stack([crop512,crop512,crop512])).unsqueeze(0).float().to(DEVICE)
        with torch.inference_mode():
            probs = np.stack([torch.sigmoid(m(x))[0,0].cpu().numpy() for m in models])
        mean_prob = probs.mean(axis=0)

        for conf_thr in CONF_TESTS:
            if det_conf < conf_thr: continue
            mask = (mean_prob > MASK_THR).astype(np.uint8)
            mask = cv2.resize(mask,(x2-x1,y2-y1),interpolation=cv2.INTER_NEAREST)
            mask[~disk[y1:y2,x1:x2]] = 0
            full = np.zeros((H,W),np.uint8); full[y1:y2,x1:x2] = mask
            full[occupied[conf_thr] > 0] = 0
            if full.sum() == 0: continue
            occupied[conf_thr][full > 0] = 1
            counters[conf_thr] += 1
            rle = maskUtils.encode(np.asfortranarray(full)); rle["counts"] = rle["counts"].decode("utf-8")
            subs[conf_thr].append({"filament_id":f"{image_id}_{counters[conf_thr]}","segmentation_rle":rle["counts"]})

for conf_thr,data in subs.items():
    df = pd.DataFrame(data, columns=["filament_id","segmentation_rle"])
    assert df["filament_id"].is_unique
    path = os.path.join(OUT_DIR,f"submission_conf2048_{str(conf_thr).replace('.','')}.csv")
    df.to_csv(path,index=False)
    print(f"conf={conf_thr}: {len(df)} instances -> {path}")

Inference: 100%|██████████| 180/180 [02:03<00:00,  1.46it/s]

conf=0.3: 1039 instances -> conf_test_submissions\submission_conf2048_03.csv


In [3]:
yaml_path = os.path.join(YOLO_DATA_DIR, "data.yaml")

from ultralytics import YOLO
import os

model = YOLO("yolo11l.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=1280,
    batch=2,
    device=0,
    workers=4,
    pretrained=True,
    optimizer="AdamW",
    lr0=1e-3,
    patience=8,
    project=os.path.join(BASE_DIR, "models"),
    name="yolo11l_adamw1e3",
    exist_ok=True,
    amp=True,
    plots=True,
    save=True
)

print(results.save_dir)

New https://pypi.org/project/ultralytics/8.4.173 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=X:\Desktop\kaggle\solar_project_cv\notebooks\yolo_dataset_l\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0,

In [4]:
import os, cv2, torch, numpy as np, pandas as pd
import segmentation_models_pytorch as smp
from ultralytics import YOLO
from pycocotools import mask as maskUtils
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_SIZE, PAD_RATIO, MASK_THR = 1280, 0.20, 0.50
OUT_DIR = "conf_test_submissions"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_DIR = r"X:\Desktop\kaggle\solar_project_cv\models"
YOLO_PATH = r"X:\Desktop\kaggle\solar_project_cv\notebooks\models\yolo11l_adamw1e3\weights\best.pt"
TEST_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\test\test_images"
CONF_TESTS = [0.30]

yolo = YOLO(YOLO_PATH)
models = []
for fold in range(5):
    m = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"best_seg_resnet34_fold_{fold}.pth"), map_location=DEVICE))
    m.eval()
    models.append(m)

subs = {c: [] for c in CONF_TESTS}
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise ValueError(path)
    if img.ndim == 3: img = img[:, :, 0] if img.shape[2] == 1 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

for fname in tqdm(sorted(os.listdir(TEST_DIR)), desc="Inference"):
    raw = load_gray(os.path.join(TEST_DIR, fname)); H, W = raw.shape
    disk = raw > 15
    clahe = cv2.createCLAHE(2.0, (8, 8))
    enhanced = clahe.apply(raw.astype(np.uint8)); enhanced[~disk] = 0
    yolo_img = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    r = yolo.predict(yolo_img, imgsz=YOLO_SIZE, conf=min(CONF_TESTS), verbose=False)[0]
    if len(r.boxes) == 0: continue
    boxes = r.boxes.xyxy.cpu().numpy(); confs = r.boxes.conf.cpu().numpy()
    order = np.argsort(-confs); image_id = os.path.splitext(fname)[0]
    occupied = {c: np.zeros((H,W), np.uint8) for c in CONF_TESTS}
    counters = {c: 0 for c in CONF_TESTS}

    for bi in order:
        det_conf = confs[bi]
        x1,y1,x2,y2 = boxes[bi].astype(int)
        bw,bh = x2-x1,y2-y1; px,py = int(bw*PAD_RATIO),int(bh*PAD_RATIO)
        x1,y1,x2,y2 = max(0,x1-px),max(0,y1-py),min(W,x2+px),min(H,y2+py)
        crop = enhanced[y1:y2,x1:x2]
        if crop.size == 0: continue

        crop512 = cv2.resize(crop,(512,512),interpolation=cv2.INTER_AREA)
        crop512 = clahe.apply(crop512).astype(np.float32)/255.0
        x = torch.from_numpy(np.stack([crop512,crop512,crop512])).unsqueeze(0).float().to(DEVICE)
        with torch.inference_mode():
            probs = np.stack([torch.sigmoid(m(x))[0,0].cpu().numpy() for m in models])
        mean_prob = probs.mean(axis=0)

        for conf_thr in CONF_TESTS:
            if det_conf < conf_thr: continue
            mask = (mean_prob > MASK_THR).astype(np.uint8)
            mask = cv2.resize(mask,(x2-x1,y2-y1),interpolation=cv2.INTER_NEAREST)
            mask[~disk[y1:y2,x1:x2]] = 0
            full = np.zeros((H,W),np.uint8); full[y1:y2,x1:x2] = mask
            full[occupied[conf_thr] > 0] = 0
            if full.sum() == 0: continue
            occupied[conf_thr][full > 0] = 1
            counters[conf_thr] += 1
            rle = maskUtils.encode(np.asfortranarray(full)); rle["counts"] = rle["counts"].decode("utf-8")
            subs[conf_thr].append({"filament_id":f"{image_id}_{counters[conf_thr]}","segmentation_rle":rle["counts"]})

for conf_thr,data in subs.items():
    df = pd.DataFrame(data, columns=["filament_id","segmentation_rle"])
    assert df["filament_id"].is_unique
    path = os.path.join(OUT_DIR,f"submission_50epoch1280_{str(conf_thr).replace('.','')}.csv")
    df.to_csv(path,index=False)
    print(f"conf={conf_thr}: {len(df)} instances -> {path}")

Inference: 100%|██████████| 180/180 [04:13<00:00,  1.41s/it]

conf=0.3: 1324 instances -> conf_test_submissions\submission_50epoch1280_03.csv
